# Music Discovery Engine — Persona Analysis

Visualises the GMM persona models fitted on top of ALS embeddings.

**Key claims this notebook supports:**
- Users are not monolithic — they have multiple distinct taste modes
- BIC-selected k varies meaningfully with listening diversity
- Persona centroids cluster into coherent taste regions across users
- Each persona within a user maps to distinct artist groups

**Sections**
1. Fleet overview — k distribution across all users
2. Persona complexity — weight balance and dominant-mode analysis
3. Taste space — UMAP of all persona centroids
4. User classification — niche vs mainstream vs diverse
5. Three case studies — deep dive into representative users
6. Artist coherence — top artists per persona within a user
7. Summary

In [1]:
import sys
sys.path.insert(0, '..')

import warnings; warnings.filterwarnings('ignore')
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.decomposition import PCA

DATA_DIR       = Path('../data/processed')
EMBEDDINGS_DIR = Path('../models/embeddings')
PERSONAS_DIR   = Path('../models/personas')
OUT_DIR        = Path('../results/persona_report')
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Setup done.')

Setup done.


In [2]:
# Load interactions + song metadata
interactions = pd.read_parquet(DATA_DIR / 'interactions.parquet')
songs_df     = pd.read_parquet(DATA_DIR / 'songs.parquet')
train        = interactions[interactions['split'] == 'train']

# Load ALS embeddings
song_emb_df  = pd.read_parquet(EMBEDDINGS_DIR / 'song_embeddings.parquet')
emb_cols     = [c for c in song_emb_df.columns if c.startswith('emb_')]
song_emb_mat = song_emb_df[emb_cols].to_numpy(dtype=np.float32)
norms        = np.linalg.norm(song_emb_mat, axis=1, keepdims=True)
song_emb_n   = song_emb_mat / np.where(norms > 0, norms, 1.0)   # unit-normalise
song_to_pos  = {sid: i for i, sid in enumerate(song_emb_df['song_id'])}
song_to_artist = dict(zip(songs_df['song_id'], songs_df['artist_name'].fillna('Unknown')))

# Load personas — only current-interaction users
current_users = set(train['user_id'].unique())
rng = np.random.default_rng(42)

personas = {}     # uid -> UserPersonaModel
for d in PERSONAS_DIR.iterdir():
    pkl = d / 'persona.pkl'
    if not pkl.exists() or d.name not in current_users:
        continue
    with open(pkl, 'rb') as f:
        p = pickle.load(f)
    personas[d.name] = p

print(f'Current-user personas loaded : {len(personas):,}')
print(f'k range                      : {min(p.k for p in personas.values())} – {max(p.k for p in personas.values())}')
print(f'Mean k                       : {np.mean([p.k for p in personas.values()]):.2f}')

Current-user personas loaded : 2,000
k range                      : 1 – 8
Mean k                       : 5.69


## 1. Fleet Overview — K Distribution

BIC selects k independently per user. The spread shows the system adapts to individual listening complexity.

In [3]:
ks          = np.array([p.k for p in personas.values()])
k_counts    = {k: int((ks == k).sum()) for k in sorted(set(ks))}
k_vals      = sorted(k_counts.keys())
k_freqs     = [k_counts[k] for k in k_vals]

# History sizes for each user
history_sizes = train.groupby('user_id')['song_id'].nunique()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# K bar chart
ax = axes[0]
bars = ax.bar(k_vals, k_freqs, color='#4C72B0', alpha=0.88, edgecolor='white')
ax.set_xlabel('Number of personas (k)')
ax.set_ylabel('Number of users')
ax.set_title('BIC-Selected k per User')
ax.set_xticks(k_vals)
for bar, freq in zip(bars, k_freqs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(freq), ha='center', va='bottom', fontsize=8)
ax.grid(axis='y', alpha=0.2)

# k vs history size scatter
ax = axes[1]
user_ks    = pd.Series({uid: p.k for uid, p in personas.items()})
user_sizes = history_sizes.reindex(user_ks.index).fillna(0)
ax.scatter(user_sizes.values, user_ks.values,
           s=4, alpha=0.3, color='#4C72B0', rasterized=True)
# Bin means
bins = np.percentile(user_sizes, np.linspace(0, 100, 11))
bins = np.unique(bins)
bin_idx  = np.digitize(user_sizes, bins) - 1
bin_k    = [user_ks.values[bin_idx == b].mean() for b in range(len(bins)-1)]
bin_mid  = [(bins[b]+bins[b+1])/2 for b in range(len(bins)-1)]
ax.plot(bin_mid, bin_k, color='red', lw=2, label='Bin mean')
ax.set_xlabel('Unique songs in training history')
ax.set_ylabel('Selected k')
ax.set_title('Listening Breadth → Persona Complexity')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# k CDF
ax = axes[2]
k_all = np.sort(ks)
cdf   = np.arange(1, len(k_all)+1) / len(k_all)
ax.step(k_all, cdf, color='#4C72B0', lw=2)
ax.set_xlabel('k')
ax.set_ylabel('Cumulative fraction of users')
ax.set_title('CDF of k')
for threshold, label in [(2, '≥2'), (3, '≥3'), (5, '≥5')]:
    frac = (ks >= threshold).mean()
    ax.axhline(1-frac, color='gray', lw=0.8, ls='--')
    ax.text(max(k_vals)-0.3, 1-frac+0.01, f'{frac*100:.0f}% have k≥{threshold}',
            ha='right', fontsize=8, color='gray')
ax.grid(alpha=0.2)

fig.suptitle(f'Persona fleet overview — {len(personas):,} users, mean k={ks.mean():.2f}',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / 'fleet_overview.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Users with k=1  (single mode)  : {(ks==1).sum():,}  ({100*(ks==1).mean():.1f}%)")
print(f"Users with k>=2 (multi-persona) : {(ks>=2).sum():,}  ({100*(ks>=2).mean():.1f}%)")
print(f"Users with k>=5                 : {(ks>=5).sum():,}  ({100*(ks>=5).mean():.1f}%)")

Users with k=1  (single mode)  : 524  (26.2%)
Users with k>=2 (multi-persona) : 1,476  (73.8%)
Users with k>=5                 : 1,390  (69.5%)


## 2. Persona Complexity — Weight Balance

For multi-persona users: are their personae equally weighted, or dominated by one mode?
High entropy = balanced, diverse listener. Low entropy = one dominant taste.

In [4]:
multi_users = {uid: p for uid, p in personas.items() if p.k >= 2}

# Weight entropy per user
def weight_entropy(w):
    w = np.asarray(w)
    return -np.sum(w * np.log(w + 1e-12))

def max_weight(p):
    return float(np.max(p.weights))

entropies   = np.array([weight_entropy(p.weights) for p in multi_users.values()])
max_weights = np.array([max_weight(p)             for p in multi_users.values()])
multi_ks    = np.array([p.k                        for p in multi_users.values()])

# Normalize entropy by log(k) so comparable across different k
norm_entropy = np.array([weight_entropy(p.weights) / np.log(p.k)
                          for p in multi_users.values()])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Normalized entropy histogram
ax = axes[0]
ax.hist(norm_entropy, bins=40, color='#55A868', edgecolor='none', alpha=0.85)
ax.axvline(norm_entropy.mean(), color='red', lw=1.5, ls='--',
           label=f'Mean = {norm_entropy.mean():.3f}')
ax.set_xlabel('Normalized weight entropy  (0 = dominated, 1 = balanced)')
ax.set_ylabel('Users')
ax.set_title('How Balanced Are Persona Weights?\n(multi-persona users only)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# Max weight by k
ax = axes[1]
for k in sorted(set(multi_ks)):
    mask = multi_ks == k
    jitter = rng.uniform(-0.2, 0.2, mask.sum())
    ax.scatter(np.full(mask.sum(), k) + jitter, max_weights[mask],
               s=6, alpha=0.3, color='#4C72B0', rasterized=True)
binned_max = [max_weights[multi_ks == k].mean() for k in sorted(set(multi_ks))]
ax.plot(sorted(set(multi_ks)), binned_max, 'r-o', lw=2, ms=5, label='Mean max weight')
ax.set_xlabel('k (number of personas)')
ax.set_ylabel('Dominant persona weight')
ax.set_title('Dominant Persona Weight vs k')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# Balanced vs dominated pie
balanced_thresh = 0.6   # dominant weight < 0.6 = balanced
balanced = (max_weights < balanced_thresh).sum()
dominated = len(max_weights) - balanced
ax = axes[2]
ax.pie([balanced, dominated],
       labels=[f'Balanced\n(max w < {balanced_thresh})', f'Dominated\n(max w ≥ {balanced_thresh})'],
       colors=['#55A868', '#DD8452'], autopct='%1.0f%%', startangle=90,
       textprops={'fontsize': 10})
ax.set_title(f'Persona Balance\n(multi-persona users, n={len(multi_users):,})')

fig.suptitle('Persona weight analysis — how users split across taste modes',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / 'persona_balance.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Multi-persona users: {len(multi_users):,}')
print(f'Balanced (max_w < {balanced_thresh}): {balanced} ({100*balanced/len(multi_users):.1f}%)')
print(f'Dominated (max_w ≥ {balanced_thresh}): {dominated} ({100*dominated/len(multi_users):.1f}%)')
print(f'Mean normalized entropy: {norm_entropy.mean():.3f}')

Multi-persona users: 1,476
Balanced (max_w < 0.6): 980 (66.4%)
Dominated (max_w ≥ 0.6): 496 (33.6%)
Mean normalized entropy: 0.672


## 3. Taste Space — UMAP of All Persona Centroids

Every persona component (GMM mean) is a point in 128-dim ALS space. Projecting all centroids to 2D reveals shared taste regions across users.

In [5]:
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print('umap-learn not installed — using PCA. pip install umap-learn')

# Collect all persona centroids
all_means   = []   # (n_total_personas, 128)
all_weights = []   # GMM weight for each centroid
all_ks      = []   # k of the user this centroid belongs to
all_uids    = []

for uid, p in personas.items():
    nrm = np.linalg.norm(p.means, axis=1, keepdims=True)
    means_n = p.means / np.where(nrm > 0, nrm, 1.0)
    all_means.append(means_n)
    all_weights.extend(p.weights.tolist())
    all_ks.extend([p.k] * p.k)
    all_uids.extend([uid] * p.k)

all_means   = np.vstack(all_means).astype(np.float32)
all_weights = np.array(all_weights)
all_ks      = np.array(all_ks)

print(f'Total persona centroids: {len(all_means):,}')

# Subsample for speed if needed
MAX_PTS = 8000
if len(all_means) > MAX_PTS:
    sub_idx   = rng.choice(len(all_means), MAX_PTS, replace=False)
    sub_means = all_means[sub_idx]
    sub_w     = all_weights[sub_idx]
    sub_ks    = all_ks[sub_idx]
else:
    sub_idx   = np.arange(len(all_means))
    sub_means = all_means
    sub_w     = all_weights
    sub_ks    = all_ks

if HAS_UMAP:
    reducer   = umap.UMAP(n_neighbors=30, min_dist=0.05, random_state=42, n_jobs=1)
    coords_2d = reducer.fit_transform(sub_means)
    proj_meth = 'UMAP'
else:
    coords_2d = PCA(n_components=2, random_state=42).fit_transform(sub_means)
    proj_meth = 'PCA'

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: coloured by GMM weight (importance of that persona component)
ax = axes[0]
sc = ax.scatter(coords_2d[:, 0], coords_2d[:, 1],
                c=sub_w, cmap='YlOrRd', s=5, alpha=0.5, rasterized=True)
plt.colorbar(sc, ax=ax, label='GMM component weight')
ax.set_title(f'{proj_meth} of all persona centroids\nColoured by component weight')
ax.set_xlabel(f'{proj_meth}-1'); ax.set_ylabel(f'{proj_meth}-2')

# Right: coloured by user k
ax = axes[1]
cmap_k = plt.cm.get_cmap('cool', max(sub_ks) - min(sub_ks) + 1)
sc2 = ax.scatter(coords_2d[:, 0], coords_2d[:, 1],
                 c=sub_ks, cmap='cool', s=5, alpha=0.5,
                 vmin=sub_ks.min(), vmax=sub_ks.max(), rasterized=True)
plt.colorbar(sc2, ax=ax, label="User's k")
ax.set_title(f'{proj_meth} of all persona centroids\nColoured by user k')
ax.set_xlabel(f'{proj_meth}-1'); ax.set_ylabel(f'{proj_meth}-2')

fig.suptitle('Global taste space: all persona centroids across all users',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / 'taste_space.png', bbox_inches='tight', dpi=150)
plt.show()

Total persona centroids: 11,381


## 4. User Classification — Niche, Mainstream, Diverse

Users classified by the spread of their song embeddings:
- **Niche**: tightly clustered history — focused, consistent taste
- **Mainstream**: moderate spread
- **Diverse**: high spread — wide-ranging listener

In [6]:
# Compute embedding spread per user (variance of unit-normalised song embeddings)
user_spread = {}
for uid in list(personas.keys()):
    songs = set(train[train['user_id'] == uid]['song_id'])
    idx   = np.array([song_to_pos[s] for s in songs if s in song_to_pos], dtype=np.intp)
    if len(idx) < 5:
        continue
    h_emb = song_emb_n[idx]
    user_spread[uid] = float(np.var(h_emb))

spread_vals = np.array(list(user_spread.values()))
low_q, high_q = np.percentile(spread_vals, 33), np.percentile(spread_vals, 66)

def classify(uid):
    s = user_spread.get(uid, np.median(spread_vals))
    if s < low_q:   return 'niche'
    if s > high_q:  return 'diverse'
    return 'mainstream'

user_types = {uid: classify(uid) for uid in personas}
type_counts = pd.Series(user_types).value_counts()

type_colors = {'niche': '#4C72B0', 'mainstream': '#DD8452', 'diverse': '#55A868'}

# Summary
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Type counts bar
ax = axes[0]
for label, cnt in type_counts.items():
    ax.bar(label, cnt, color=type_colors[label], alpha=0.88, edgecolor='white')
    ax.text(label, cnt + 5, str(cnt), ha='center', fontsize=9)
ax.set_ylabel('Users')
ax.set_title('User Type Distribution')
ax.grid(axis='y', alpha=0.2)

# Mean k per type
type_k = {}
for uid, p in personas.items():
    t = user_types.get(uid)
    if t:
        type_k.setdefault(t, []).append(p.k)

ax = axes[1]
type_order = ['niche', 'mainstream', 'diverse']
ax.bar([t.title() for t in type_order],
       [np.mean(type_k[t]) for t in type_order],
       color=[type_colors[t] for t in type_order], alpha=0.88, edgecolor='white')
ax.set_ylabel('Mean k (number of personas)')
ax.set_title('Mean k by User Type\nDiverse users get more personas')
for i, t in enumerate(type_order):
    ax.text(i, np.mean(type_k[t]) + 0.05,
            f'{np.mean(type_k[t]):.2f}', ha='center', fontsize=9)
ax.grid(axis='y', alpha=0.2)

# Embedding spread distribution by type
ax = axes[2]
for t in type_order:
    vals = [user_spread[uid] for uid in user_types if user_types[uid] == t and uid in user_spread]
    ax.hist(vals, bins=30, alpha=0.6, color=type_colors[t], label=t.title())
ax.set_xlabel('Embedding variance (listening spread)')
ax.set_ylabel('Users')
ax.set_title('Spread Distribution by Type')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

fig.suptitle('User taste classification via listening-history embedding spread',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / 'user_classification.png', bbox_inches='tight', dpi=150)
plt.show()
for t in type_order:
    mk = np.mean(type_k[t])
    print(f'{t:12s}: n={type_counts.get(t,0):4d}, mean k={mk:.2f}')

niche       : n= 610, mean k=6.09
mainstream  : n= 761, mean k=5.14
diverse     : n= 629, mean k=5.96


## 5. Three Case Studies

One representative user from each type: niche, mainstream, diverse.
Shows how the GMM adapts its persona structure to different listening patterns.

In [7]:
def pick_rep(utype, min_k=1, min_songs=20):
    """Pick the most representative user (spread closest to type median)."""
    candidates = [
        uid for uid, t in user_types.items()
        if t == utype and personas[uid].k >= min_k
        and uid in user_spread
    ]
    if not candidates:
        return None
    type_median = np.median([user_spread[u] for u in candidates])
    return min(candidates, key=lambda u: abs(user_spread[u] - type_median))

rep_users = {
    'niche':       pick_rep('niche',       min_k=2),
    'mainstream':  pick_rep('mainstream',  min_k=2),
    'diverse':     pick_rep('diverse',     min_k=3),
}
print('Representative users:', {t: uid[:16] if uid else None for t, uid in rep_users.items()})

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for col, (utype, uid) in enumerate(rep_users.items()):
    if uid is None:
        axes[0, col].set_axis_off(); axes[1, col].set_axis_off()
        continue

    p = personas[uid]
    songs = set(train[train['user_id'] == uid]['song_id'])
    hist_idx  = np.array([song_to_pos[s] for s in songs if s in song_to_pos], dtype=np.intp)
    hist_emb  = song_emb_n[hist_idx]
    hist_sids = [s for s in songs if s in song_to_pos]
    assignments = p.dominant_persona(hist_emb)

    # 2D projection
    if HAS_UMAP and len(hist_emb) >= 10:
        h2d = umap.UMAP(n_neighbors=min(15, len(hist_emb)-1),
                        min_dist=0.08, random_state=42, n_jobs=1).fit_transform(hist_emb)
    else:
        h2d = PCA(n_components=2, random_state=42).fit_transform(hist_emb)

    colors_p = plt.cm.get_cmap('Set1', p.k)

    # Top: songs coloured by persona
    ax = axes[0, col]
    for ki in range(p.k):
        mask = assignments == ki
        ax.scatter(h2d[mask, 0], h2d[mask, 1],
                   s=18, alpha=0.8, color=colors_p(ki), label=f'P{ki+1} (w={p.weights[ki]:.2f})')
    ax.set_title(f'{utype.title()} user  |  k={p.k}\n{len(hist_idx)} songs', fontsize=11)
    ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
    ax.legend(fontsize=7.5, loc='best')
    ax.grid(alpha=0.18)

    # Bottom: persona weight bars
    ax = axes[1, col]
    bar_colors = [colors_p(ki) for ki in range(p.k)]
    ax.bar([f'P{ki+1}' for ki in range(p.k)], p.weights,
           color=bar_colors, alpha=0.88, edgecolor='white')
    ax.set_ylim(0, 1)
    ax.set_xlabel('Persona component')
    ax.set_ylabel('GMM weight')
    ax.set_title(f'Component weights\n(spread = {user_spread[uid]:.5f})')
    ax.grid(axis='y', alpha=0.2)
    for ki, w in enumerate(p.weights):
        ax.text(ki, w + 0.01, f'{w:.2f}', ha='center', fontsize=8.5)

fig.suptitle('Case studies: niche / mainstream / diverse user personas',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / 'case_studies.png', bbox_inches='tight', dpi=150)
plt.show()

Representative users: {'niche': 'e66bcf8f1c2bb171', 'mainstream': '70b3d15a780eb75a', 'diverse': 'd84d5e81fc5c0308'}


## 6. Artist Coherence per Persona

For the diverse case-study user: top artists within each persona component.
Shows the system groups musically coherent artists together without any genre labels.

In [8]:
def top_artists_per_persona(uid, p, top_n=6):
    songs  = list(train[train['user_id'] == uid]['song_id'])
    idx    = np.array([song_to_pos[s] for s in songs if s in song_to_pos], dtype=np.intp)
    sids   = [s for s in songs if s in song_to_pos]
    if len(idx) == 0:
        return {}
    embs   = song_emb_n[idx]
    assign = p.dominant_persona(embs)
    result = {}
    for ki in range(p.k):
        mask    = assign == ki
        p_songs = [sids[j] for j in range(len(sids)) if mask[j]]
        artists = [song_to_artist.get(s, 'Unknown') for s in p_songs]
        counts  = pd.Series(artists).value_counts().head(top_n)
        result[ki] = counts
    return result

# Use diverse user for most interesting result
target_uid = rep_users.get('diverse') or rep_users.get('mainstream')
if target_uid is None:
    print('No representative user found.')
else:
    target_p = personas[target_uid]
    top_artists = top_artists_per_persona(target_uid, target_p)

    n_cols  = min(target_p.k, 4)
    n_rows  = (target_p.k + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5.5 * n_cols, 3.5 * n_rows),
                             squeeze=False)
    colors_p = plt.cm.get_cmap('Set1', target_p.k)

    for ki in range(target_p.k):
        r, c = divmod(ki, n_cols)
        ax   = axes[r][c]
        counts = top_artists.get(ki, pd.Series(dtype=int))
        if counts.empty:
            ax.set_axis_off(); continue
        y_pos = np.arange(len(counts))
        ax.barh(y_pos, counts.values, color=colors_p(ki), alpha=0.88, edgecolor='white')
        ax.set_yticks(y_pos)
        ax.set_yticklabels(counts.index, fontsize=8.5)
        ax.invert_yaxis()
        ax.set_xlabel('Songs')
        ax.set_title(f'Persona {ki+1}  (w={target_p.weights[ki]:.2f},  n={counts.sum()})',
                     fontsize=10)
        ax.grid(axis='x', alpha=0.2)

    # Hide unused subplots
    for ki in range(target_p.k, n_rows * n_cols):
        r, c = divmod(ki, n_cols)
        axes[r][c].set_axis_off()

    fig.suptitle(f'Top artists per persona — diverse user ({target_uid[:16]}...)\n'
                 f'k={target_p.k} components, coloured by persona',
                 fontsize=12, fontweight='bold')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'artist_coherence.png', bbox_inches='tight', dpi=150)
    plt.show()

## 7. Intra-User Persona Separation

For multi-persona users, how distinct are the persona centroids from each other?
High pairwise distance = the model found genuinely different taste clusters.

In [9]:
from itertools import combinations

within_dists, k_list = [], []
for uid, p in personas.items():
    if p.k < 2:
        continue
    nrm    = np.linalg.norm(p.means, axis=1, keepdims=True)
    means_n = p.means / np.where(nrm > 0, nrm, 1.0)
    dists  = [
        1.0 - float(means_n[i] @ means_n[j])
        for i, j in combinations(range(p.k), 2)
    ]
    within_dists.append(np.mean(dists))
    k_list.append(p.k)

within_dists = np.array(within_dists)
k_list       = np.array(k_list)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.hist(within_dists, bins=40, color='#8172B2', alpha=0.85, edgecolor='none')
ax.axvline(within_dists.mean(), color='red', lw=1.5, ls='--',
           label=f'Mean = {within_dists.mean():.3f}')
ax.set_xlabel('Mean pairwise cosine distance between persona centroids')
ax.set_ylabel('Users')
ax.set_title('Persona Separation within Users\n(higher = more distinct taste modes)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

ax = axes[1]
for k in sorted(set(k_list)):
    mask    = k_list == k
    jitter  = rng.uniform(-0.15, 0.15, mask.sum())
    ax.scatter(np.full(mask.sum(), k) + jitter, within_dists[mask],
               s=5, alpha=0.3, rasterized=True)
binned_d = [within_dists[k_list == k].mean() for k in sorted(set(k_list))]
ax.plot(sorted(set(k_list)), binned_d, 'r-o', lw=2, ms=5, label='Bin mean')
ax.set_xlabel('k')
ax.set_ylabel('Mean pairwise cosine distance')
ax.set_title('Separation vs k\n(more personas → slightly lower average separation)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

fig.suptitle('Persona centroids are well-separated within users',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / 'persona_separation.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Mean intra-user centroid distance : {within_dists.mean():.4f}')
print(f'% users with mean dist > 0.5      : {100*(within_dists > 0.5).mean():.1f}%')

Mean intra-user centroid distance : 0.7175
% users with mean dist > 0.5      : 92.1%


## 8. Summary

In [10]:
multi_mask = ks >= 2

print('=' * 58)
print('PERSONA MODEL SUMMARY')
print('=' * 58)
print(f'  Total users with personas : {len(personas):,}')
print(f'  k range                   : {ks.min()} – {ks.max()}')
print(f'  Mean k                    : {ks.mean():.2f}')
print(f'  Users with k = 1          : {(ks==1).sum():,}  ({100*(ks==1).mean():.1f}%)')
print(f'  Users with k ≥ 2          : {(ks>=2).sum():,}  ({100*(ks>=2).mean():.1f}%)')
print()
if len(multi_users) > 0:
    print(f'  Mean weight entropy (norm): {norm_entropy.mean():.3f}  (1.0 = perfectly balanced)')
    print(f'  Balanced users (max_w<0.6): {balanced:,}  ({100*balanced/len(multi_users):.1f}% of multi-k)')
print()
if len(within_dists) > 0:
    print(f'  Mean centroid separation  : {within_dists.mean():.4f}')
    print(f'  Centroids dist > 0.5      : {100*(within_dists>0.5).mean():.1f}% of multi-k users')
print()
for utype in ['niche', 'mainstream', 'diverse']:
    n    = type_counts.get(utype, 0)
    mk   = np.mean(type_k.get(utype, [0]))
    print(f'  {utype:12s}: {n:4d} users, mean k = {mk:.2f}')
print()
print('Saved plots:')
for p in [
    OUT_DIR / 'fleet_overview.png',
    OUT_DIR / 'persona_balance.png',
    OUT_DIR / 'taste_space.png',
    OUT_DIR / 'user_classification.png',
    OUT_DIR / 'case_studies.png',
    OUT_DIR / 'artist_coherence.png',
    OUT_DIR / 'persona_separation.png',
]:
    print(f"  {'✓' if p.exists() else '○'}  {p}")

PERSONA MODEL SUMMARY
  Total users with personas : 2,000
  k range                   : 1 – 8
  Mean k                    : 5.69
  Users with k = 1          : 524  (26.2%)
  Users with k ≥ 2          : 1,476  (73.8%)

  Mean weight entropy (norm): 0.672  (1.0 = perfectly balanced)
  Balanced users (max_w<0.6): 980  (66.4% of multi-k)

  Mean centroid separation  : 0.7175
  Centroids dist > 0.5      : 92.1% of multi-k users

  niche       :  610 users, mean k = 6.09
  mainstream  :  761 users, mean k = 5.14
  diverse     :  629 users, mean k = 5.96

Saved plots:
  ✓  ../results/persona_report/fleet_overview.png
  ✓  ../results/persona_report/persona_balance.png
  ✓  ../results/persona_report/taste_space.png
  ✓  ../results/persona_report/user_classification.png
  ✓  ../results/persona_report/case_studies.png
  ✓  ../results/persona_report/artist_coherence.png
  ✓  ../results/persona_report/persona_separation.png
